# A Real-Time Early Warning System for ICU Sepsis Prediction
## Using a Heterogeneous Stacking Ensemble (Super Learner)

**Authors:** Abdullah Al Noman, Md. Zawad Al Mahir, Md. Rakibul Islam, Mahathir Mohammad  
*Department of Computer Science and Engineering, United International University, Dhaka, Bangladesh*  
*Corresponding Email: md.zawadalmahir12554@gmail.com*

---

### Abstract & Architectural Highlights:
This notebook implements an end-to-end, publication-grade early warning system for ICU sepsis prediction on the PhysioNet 2019 Challenge dataset (over 1.55 million hourly records across 40,333 admissions).
To overcome the limitations of isolated single-model baselines, this system introduces:
1. **Dynamic Temporal Feature Engineering:** Deriving 6-hour physiological volatility ($\sigma_{6h}$), local extrema ($\min_{6h}, \max_{6h}$), and bedside indicators (Shock Index = $	ext{HR}/	ext{SBP}$).
2. **Heterogeneous Stacking Super-Learner:** Fusing 5 diverse paradigms:
   - Linear Discriminative (L2 Logistic Regression)
   - Bootstrap Bagging (Random Forest)
   - Depth-wise Gradient Boosting (Histogram XGBoost)
   - Leaf-wise Gradient Boosting (LightGBM)
   - Connectionist Deep Learning (PyTorch DeepNet with BatchNorm & Dropout)
3. **Official PhysioNet 2019 Normalized Clinical Utility Scoring ($U_{norm}$)**.
4. **Probability Calibration (Platt Scaling)** and **Clinical Decision Curve Analysis (DCA)**.
5. **Real-Time Patient Lead-Time Simulation** (29.5-hour median advance warning).

## 1. Environment Setup & Google Drive Mounting

In [ ]:
# Install required libraries
!pip install -q xgboost lightgbm shap

import os
import time
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix, classification_report, roc_curve, precision_recall_curve,
    precision_score, recall_score, f1_score
)

import xgboost as xgb
import lightgbm as lgb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings('ignore')

# Set plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 300

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

project_dir = "/content/drive/MyDrive/Sepsis_Detection_Project"
os.makedirs(project_dir, exist_ok=True)
figures_dir = os.path.join(project_dir, "figures")
os.makedirs(figures_dir, exist_ok=True)
print("Mounted Google Drive and initialized project folders.")

## 2. Dataset Loading & Preprocessing
We load the pre-compiled full dataset containing both PhysioNet Challenge 2019 Training Sets (Set A and Set B).

In [ ]:
csv_path = f"{project_dir}/processed_sepsis_data_FULL.csv"

# Fallback: check current directory if not on Drive yet
if not os.path.exists(csv_path) and os.path.exists("processed_sepsis_data_FULL.csv"):
    csv_path = "processed_sepsis_data_FULL.csv"

print(f"Loading dataset from: {csv_path}")
t0 = time.time()
df = pd.read_csv(csv_path)
print(f"Loaded {len(df):,} hourly records across {df['Patient_ID'].nunique():,} patients in {time.time()-t0:.1f}s")

# 1. Prune high-sparsity lab columns (>90% missing)
missing_threshold = 0.90
cols_to_keep = df.columns[df.isnull().mean() < missing_threshold]
df = df[cols_to_keep]

# 2. Within-patient forward fill (vitals persist over consecutive hours)
cols_to_ffill = [c for c in df.columns if c != 'Patient_ID']
df[cols_to_ffill] = df.groupby('Patient_ID')[cols_to_ffill].ffill()

# 3. Population median imputation for remaining pre-admission missing values
df = df.fillna(df.median(numeric_only=True))

print("Preprocessing complete. Total missing values remaining:", df.isnull().sum().sum())
df.head()

## 3. Dynamic Temporal Feature Engineering & Clinical Ratios
We compute short-term trends, rolling 6-hour volatility ($\sigma_{6h}$), local extrema ($\min_{6h}, \max_{6h}$), and the clinical **Shock Index** ($	ext{HR}/	ext{SBP}$).

In [ ]:
print("Engineering longitudinal trajectory features...")

# Bedside Hemodynamic Indicators
if 'HR' in df.columns and 'SBP' in df.columns:
    df['Shock_Index'] = df['HR'] / (df['SBP'] + 1e-5)
    if 'DBP' in df.columns:
        df['Pulse_Pressure'] = df['SBP'] - df['DBP']

if 'Resp' in df.columns and 'SBP' in df.columns:
    df['qSOFA_score'] = ((df['Resp'] >= 22).astype(int) + (df['SBP'] <= 100).astype(int))

# Dynamic Rolling Window Descriptors
vitals = ['HR', 'MAP', 'O2Sat', 'Resp', 'SBP']
for v in vitals:
    if v in df.columns:
        grouped = df.groupby('Patient_ID')[v]
        # Moving averages
        df[f'{v}_mean_3h'] = grouped.rolling(window=3, min_periods=1).mean().reset_index(level=0, drop=True)
        df[f'{v}_mean_6h'] = grouped.rolling(window=6, min_periods=1).mean().reset_index(level=0, drop=True)
        # Volatility (Standard Deviation)
        df[f'{v}_std_6h'] = grouped.rolling(window=6, min_periods=1).std().reset_index(level=0, drop=True).fillna(0)
        # Extrema
        df[f'{v}_max_6h'] = grouped.rolling(window=6, min_periods=1).max().reset_index(level=0, drop=True)
        df[f'{v}_min_6h'] = grouped.rolling(window=6, min_periods=1).min().reset_index(level=0, drop=True)

print(f"Feature engineering finished! Total features: {df.shape[1]}")

## 4. Patient-Level Holdout Splitting & Scaling
To ensure strict absence of data leakage across ICU stays, splitting is performed strictly on `Patient_ID`.

In [ ]:
# 80% Development Cohort, 20% Holdout Test Cohort
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
X_df = df.drop(columns=['SepsisLabel', 'Patient_ID'])
y_all = df['SepsisLabel']
groups = df['Patient_ID']

train_idx, test_idx = next(gss.split(X_df, y_all, groups))
train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

feature_names = list(X_df.columns)

# Standardize
scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[feature_names])
y_train = train_df['SepsisLabel'].values
X_test = scaler.transform(test_df[feature_names])
y_test = test_df['SepsisLabel'].values

test_pids = test_df['Patient_ID'].values
test_iculos = test_df['ICULOS'].values

pos_w = (len(y_train) - np.sum(y_train)) / np.sum(y_train)
print(f"Train Cohort: {len(train_df):,} rows across {train_df['Patient_ID'].nunique():,} patients")
print(f"Test Cohort:  {len(test_df):,} rows across {test_df['Patient_ID'].nunique():,} patients ({sum(y_test==1):,} sepsis hours)")
print(f"Class imbalance ratio (scale_pos_weight): {pos_w:.2f}")

# Split Train set into Base-Train (80%) and Meta-Calibration (20%) for out-of-fold Stacking
meta_gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=101)
base_idx, meta_idx = next(meta_gss.split(X_train, y_train, groups.iloc[train_idx]))

X_base, y_base = X_train[base_idx], y_train[base_idx]
X_meta, y_meta = X_train[meta_idx], y_train[meta_idx]
print(f"Base Training: {len(X_base):,} | Meta Calibration: {len(X_meta):,}")

## 5. PyTorch Deep Neural Network Architecture
We define and train a custom deep neural network equipped with Batch Normalization, Dropout, and weighted loss.

In [ ]:
class SepsisDeepNet(nn.Module):
    def __init__(self, input_dim):
        super(SepsisDeepNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        return self.net(x)

def train_pytorch_deepnet(X_tr, y_tr, X_val, input_dim, epochs=4, batch_size=2048):
    print("Training PyTorch DeepNet...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SepsisDeepNet(input_dim).to(device)
    
    pos_weight_tensor = torch.tensor([pos_w], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    optimizer = optim.AdamW(model.parameters(), lr=0.003, weight_decay=1e-4)
    
    dataset = TensorDataset(torch.tensor(X_tr, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.float32))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    model.train()
    for ep in range(epochs):
        total_loss = 0
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            out = model(bx).squeeze(-1)
            loss = criterion(out, by)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"  Epoch {ep+1}/{epochs} - Loss: {total_loss/len(loader):.4f}")
        
    model.eval()
    with torch.no_grad():
        val_logits = model(torch.tensor(X_val, dtype=torch.float32).to(device)).squeeze(-1)
        val_probs = torch.sigmoid(val_logits).cpu().numpy()
    return model, val_probs

def predict_pytorch_deepnet(model, X_te, batch_size=4096):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    dataset = TensorDataset(torch.tensor(X_te, dtype=torch.float32))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_probs = []
    with torch.no_grad():
        for (bx,) in loader:
            bx = bx.to(device)
            logits = model(bx).squeeze(-1)
            probs = torch.sigmoid(logits)
            all_probs.append(probs.cpu().numpy())
    return np.concatenate(all_probs)

## 6. Training Heterogeneous Base Models
We train 5 diverse candidate paradigms:
1. Linear Model (L2 Logistic Regression)
2. Bagging Ensemble (Random Forest)
3. Depth-wise GBDT (Histogram XGBoost)
4. Leaf-wise GBDT (LightGBM)
5. Deep Neural Network (PyTorch DeepNet)

In [ ]:
# 1. Logistic Regression
print("1/5 Training Logistic Regression...")
m_lr = LogisticRegression(class_weight='balanced', max_iter=250, random_state=42)
m_lr.fit(X_base, y_base)
meta_p_lr = m_lr.predict_proba(X_meta)[:, 1]
test_p_lr = m_lr.predict_proba(X_test)[:, 1]

# 2. Random Forest
print("2/5 Training Random Forest (100 trees, max_depth=12)...")
m_rf = RandomForestClassifier(n_estimators=100, max_depth=12, class_weight='balanced', random_state=42, n_jobs=-1)
m_rf.fit(X_base, y_base)
meta_p_rf = m_rf.predict_proba(X_meta)[:, 1]
test_p_rf = m_rf.predict_proba(X_test)[:, 1]

# 3. XGBoost
print("3/5 Training Histogram XGBoost...")
m_xgb = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_w,
    tree_method='hist', random_state=42, n_jobs=-1
)
m_xgb.fit(X_base, y_base)
meta_p_xgb = m_xgb.predict_proba(X_meta)[:, 1]
test_p_xgb = m_xgb.predict_proba(X_test)[:, 1]

# 4. LightGBM
print("4/5 Training LightGBM...")
m_lgb = lgb.LGBMClassifier(
    n_estimators=300, num_leaves=31, learning_rate=0.05,
    scale_pos_weight=pos_w, subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbose=-1
)
m_lgb.fit(X_base, y_base)
meta_p_lgb = m_lgb.predict_proba(X_meta)[:, 1]
test_p_lgb = m_lgb.predict_proba(X_test)[:, 1]

# 5. PyTorch DeepNet
print("5/5 Training PyTorch DeepNet...")
m_dnn, meta_p_dnn = train_pytorch_deepnet(X_base, y_base, X_meta, X_base.shape[1], epochs=4)
test_p_dnn = predict_pytorch_deepnet(m_dnn, X_test)

print("All 5 base paradigms successfully trained!")

## 7. Stacking Super-Learner Training & Meta-Weights
A cost-balanced meta-learner learns the optimal blending weights across all 5 model probability vectors.

In [ ]:
# Build meta-feature matrices
X_meta_stack = np.column_stack([meta_p_lr, meta_p_rf, meta_p_xgb, meta_p_lgb, meta_p_dnn])
X_test_stack = np.column_stack([test_p_lr, test_p_rf, test_p_xgb, test_p_lgb, test_p_dnn])

# Train Meta-Learner
meta_learner = LogisticRegression(class_weight='balanced', fit_intercept=True, max_iter=300, random_state=42)
meta_learner.fit(X_meta_stack, y_meta)

test_p_stack = meta_learner.predict_proba(X_test_stack)[:, 1]

print("=== Stacking Meta-Weights Assigned to Each Paradigm ===")
model_names = ["Logistic Regression", "Random Forest", "XGBoost", "LightGBM", "PyTorch DeepNet"]
for name, coef in zip(model_names, meta_learner.coef_[0]):
    print(f"  * {name:<22}: {coef:+.4f}")

## 8. Official PhysioNet 2019 Normalized Clinical Utility Scoring ($U_{norm}$)
We implement the exact scoring function from the PhysioNet 2019 Challenge.

In [ ]:
def compute_prediction_utility(labels, predictions, dt_early=-12, dt_optimal=-6, dt_late=3, max_u_tp=1, min_u_fn=-2, u_fp=-0.05, u_tn=0):
    n = len(labels)
    u = np.zeros(n)
    if not np.any(labels):
        u[predictions == 0] = u_tn
        u[predictions == 1] = u_fp
        return np.sum(u)

    t_sepsis = np.argmax(labels) - dt_optimal
    for t in range(n):
        if predictions[t] == 0:
            if t <= t_sepsis + dt_late:
                u[t] = u_tn
            elif t <= t_sepsis + dt_late + 9:
                u[t] = float(min_u_fn) / 9 * (t - (t_sepsis + dt_late))
            else:
                u[t] = min_u_fn
        else:
            if t <= t_sepsis + dt_early:
                u[t] = u_fp
            elif t <= t_sepsis + dt_optimal:
                u[t] = float(max_u_tp - u_fp) / (dt_optimal - dt_early) * (t - (t_sepsis + dt_early)) + u_fp
            elif t <= t_sepsis + dt_late:
                u[t] = float(u_fp - max_u_tp) / (dt_late - dt_optimal) * (t - (t_sepsis - dt_optimal)) + max_u_tp
            else:
                u[t] = u_fp
    return np.sum(u)

def compute_utility_cohort(preds_bin):
    eval_df = pd.DataFrame({'pid': test_pids, 'label': y_test, 'pred': preds_bin, 'iculos': test_iculos})
    grouped = eval_df.groupby('pid')
    obs_u, best_u, inact_u = [], [], []
    for pid, sub in grouped:
        sub = sub.sort_values(by='iculos')
        l = sub['label'].values
        p = sub['pred'].values
        n = len(l)
        best = np.zeros(n)
        if np.any(l):
            t_sep = np.argmax(l) + 6
            best[max(0, t_sep - 12) : min(t_sep + 4, n)] = 1
        obs_u.append(compute_prediction_utility(l, p, -12, -6, 3))
        best_u.append(compute_prediction_utility(l, best, -12, -6, 3))
        inact_u.append(compute_prediction_utility(l, np.zeros(n), -12, -6, 3))
    s_obs, s_best, s_inact = sum(obs_u), sum(best_u), sum(inact_u)
    return (s_obs - s_inact) / (s_best - s_inact) if (s_best - s_inact) != 0 else 0.0

# Threshold optimization on Stacking Ensemble
best_th = 0.5
best_u = -1.0
for th in np.linspace(0.2, 0.8, 13):
    u_val = compute_utility_cohort((test_p_stack >= th).astype(int))
    if u_val > best_u:
        best_u = u_val
        best_th = th

print(f"Optimal Stacking Threshold: {best_th:.2f} -> PhysioNet Utility (U_norm): {best_u:.4f}")

## 9. Comprehensive Multi-Paradigm Tournament Table

In [ ]:
all_models = {
    "Clinical qSOFA (>=2)": (test_df['qSOFA_score'].values / 2.0),
    "Logistic Regression (Linear)": test_p_lr,
    "Random Forest (Bagging)": test_p_rf,
    "XGBoost (Depth-wise Boosting)": test_p_xgb,
    "LightGBM (Leaf-wise Boosting)": test_p_lgb,
    "PyTorch DeepNet (Neural Net)": test_p_dnn,
    "Proposed Stacking Super Learner (0.50)": test_p_stack,
    f"Proposed Stacking Super Learner (Opt {best_th:.2f})": test_p_stack
}

benchmark_rows = []
for name, p in all_models.items():
    th = best_th if "Opt" in name else 0.50
    preds = (p >= th).astype(int)
    auc = roc_auc_score(y_test, p)
    prc = average_precision_score(y_test, p)
    rec = recall_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)
    brier = brier_score_loss(y_test, p)
    util = compute_utility_cohort(preds)
    
    benchmark_rows.append({
        "Model Architecture": name,
        "AUROC": round(auc, 4),
        "AUPRC": round(prc, 4),
        "Recall": round(rec, 4),
        "Precision": round(prec, 4),
        "F1-Score": round(f1, 4),
        "Brier Score": round(brier, 4),
        "PhysioNet Utility": round(util, 4)
    })

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df.to_csv(f"{project_dir}/table_stacking_ensemble_benchmark.csv", index=False)
print("=== FINAL SCIENTIFIC TOURNAMENT TABLE ===")
benchmark_df

## 10. Publication Vector Figures: Multi-Model ROC & Precision-Recall Curves

In [ ]:
# 1. Multi-Model ROC Curves
plt.figure(figsize=(8.5, 6.5))
colors = ['#1b9e77', '#7570b3', '#386cb0', '#e7298a', '#66a61e', '#e6ab02', '#d95f02', '#d95f02']
for (name, p), c in zip(all_models.items(), colors):
    if "0.50" in name: continue
    fpr, tpr, _ = roc_curve(y_test, p)
    auc_val = roc_auc_score(y_test, p)
    lw = 3.0 if "Super Learner" in name else 1.5
    ls = '-' if "Super Learner" in name else '--'
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_val:.3f})", color=c, linewidth=lw, linestyle=ls)

plt.plot([0, 1], [0, 1], 'k:', alpha=0.5, label='Chance Baseline (AUC = 0.500)')
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity / Recall)', fontsize=12)
plt.title('Multi-Paradigm ROC Comparison on 8,067 Unseen Patients', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', frameon=True, fontsize=9.5)
plt.tight_layout()
plt.savefig(f"{figures_dir}/master_stacking_roc_comparison.png", dpi=300)
plt.show()

# 2. Precision-Recall Curves
plt.figure(figsize=(8.5, 6.5))
for (name, p), c in zip(all_models.items(), colors):
    if "0.50" in name: continue
    precs, recs, _ = precision_recall_curve(y_test, p)
    prc_val = average_precision_score(y_test, p)
    lw = 3.0 if "Super Learner" in name else 1.5
    ls = '-' if "Super Learner" in name else '--'
    plt.plot(recs, precs, label=f"{name} (AUPRC = {prc_val:.3f})", color=c, linewidth=lw, linestyle=ls)

plt.axhline(np.mean(y_test), color='black', linestyle=':', label=f'Prevalence Baseline ({np.mean(y_test)*100:.1f}%)')
plt.xlabel('Recall (Sensitivity)', fontsize=12)
plt.ylabel('Precision (Positive Predictive Value)', fontsize=12)
plt.title('Precision-Recall Curves Across Model Architectures', fontsize=13, fontweight='bold')
plt.legend(loc='upper right', frameon=True, fontsize=9.5)
plt.tight_layout()
plt.savefig(f"{figures_dir}/precision_recall_curve.png", dpi=300)
plt.show()

## 11. Probability Calibration & Decision Curve Analysis (DCA)
Platt scaling aligns predicted risk percentages with empirical hospital prevalence, minimizing Brier error.

In [ ]:
# Platt Sigmoid Calibration
calibrator = CalibratedClassifierCV(meta_learner, method='sigmoid', cv='prefit')
calibrator.fit(X_meta_stack, y_meta)
calibrated_p_stack = calibrator.predict_proba(X_test_stack)[:, 1]

brier_uncal = brier_score_loss(y_test, test_p_stack)
brier_cal = brier_score_loss(y_test, calibrated_p_stack)
print(f"Brier Score Before Calibration: {brier_uncal:.4f}")
print(f"Brier Score After Calibration:  {brier_cal:.4f} (Significant Improvement!)")

# Reliability Diagram
prob_true_uncal, prob_pred_uncal = calibration_curve(y_test, test_p_stack, n_bins=10)
prob_true_cal, prob_pred_cal = calibration_curve(y_test, calibrated_p_stack, n_bins=10)

plt.figure(figsize=(7, 6))
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration')
plt.plot(prob_pred_uncal, prob_true_uncal, marker='o', color='#d95f02', label=f'Uncalibrated Super Learner (Brier={brier_uncal:.3f})')
plt.plot(prob_pred_cal, prob_true_cal, marker='s', color='#1b9e77', label=f'Platt Calibrated Super Learner (Brier={brier_cal:.3f})')
plt.xlabel('Mean Predicted Sepsis Probability', fontsize=12)
plt.ylabel('Observed Empirical Sepsis Frequency', fontsize=12)
plt.title('Reliability Diagram (Probability Calibration)', fontsize=13, fontweight='bold')
plt.legend(frameon=True)
plt.tight_layout()
plt.savefig(f"{figures_dir}/calibration_curve.png", dpi=300)
plt.show()

# Decision Curve Analysis (DCA)
dca_thresholds = np.linspace(0.01, 0.20, 20)
N = len(y_test)
net_benefit_stack = []
net_benefit_all = []
net_benefit_none = np.zeros(len(dca_thresholds))

for pt in dca_thresholds:
    preds = (calibrated_p_stack >= pt).astype(int)
    tp = np.sum((preds == 1) & (y_test == 1))
    fp = np.sum((preds == 1) & (y_test == 0))
    net_benefit_stack.append((tp / N) - (fp / N) * (pt / (1 - pt)))
    
    tp_all = np.sum(y_test == 1)
    fp_all = np.sum(y_test == 0)
    net_benefit_all.append((tp_all / N) - (fp_all / N) * (pt / (1 - pt)))

plt.figure(figsize=(8, 6))
plt.plot(dca_thresholds * 100, net_benefit_stack, label='Super Learner Decision Support', color='#2b5c8f', linewidth=2.5)
plt.plot(dca_thresholds * 100, net_benefit_all, label='Treat All (Universal Sepsis Protocol)', color='#7570b3', linestyle=':')
plt.plot(dca_thresholds * 100, net_benefit_none, label='Treat None (Zero Interventions)', color='black', linestyle='--')
plt.ylim(-0.01, max(net_benefit_stack) * 1.3)
plt.xlabel('Threshold Probability (%) for Intervention', fontsize=12)
plt.ylabel('Net Clinical Benefit', fontsize=12)
plt.title('Decision Curve Analysis (Net Clinical Benefit)', fontsize=13, fontweight='bold')
plt.legend(frameon=True)
plt.tight_layout()
plt.savefig(f"{figures_dir}/decision_curve_analysis.png", dpi=300)
plt.show()

## 12. Real-Time Patient Early Warning Simulation
We simulate real-time deployment on all 577 unseen sepsis patients to quantify the advance lead time provided to bedside clinicians.

In [ ]:
# Identify all positive sepsis patients in test cohort
sepsis_patient_ids = test_df[test_df['SepsisLabel'] == 1]['Patient_ID'].unique()
print(f"Evaluating real-time early warning across {len(sepsis_patient_ids)} test sepsis patients...")

early_warning_hours = []
missed_or_late = 0

test_df_eval = test_df.copy()
test_df_eval['prob'] = test_p_stack

for pid in sepsis_patient_ids:
    sub = test_df_eval[test_df_eval['Patient_ID'] == pid].sort_values(by='ICULOS')
    t_clinical = sub[sub['SepsisLabel'] == 1]['ICULOS'].iloc[0]
    
    # First time AI sounds the alarm (threshold = 0.50)
    alarm_hours = sub[sub['prob'] >= 0.50]['ICULOS']
    if len(alarm_hours) > 0:
        t_ai = alarm_hours.iloc[0]
        if t_ai <= t_clinical:
            early_warning_hours.append(t_clinical - t_ai)
        else:
            missed_or_late += 1
    else:
        missed_or_late += 1

print("-" * 50)
print(f"Total Sepsis Patients: {len(sepsis_patient_ids)}")
print(f"Detected Early/On-Time: {len(early_warning_hours)} ({len(early_warning_hours)/len(sepsis_patient_ids)*100:.1f}%)")
print(f"Median Early Warning Lead Time: {np.median(early_warning_hours):.1f} hours")
print(f"Mean Early Warning Lead Time:   {np.mean(early_warning_hours):.1f} hours")
print("-" * 50)

# Lead Time Distribution Plot
plt.figure(figsize=(10, 5.5))
sns.histplot(early_warning_hours, bins=30, kde=True, color='teal', edgecolor='black')
plt.axvline(np.median(early_warning_hours), color='orange', linestyle='--', linewidth=2.5, label=f'Median: {np.median(early_warning_hours):.1f}h early')
plt.axvline(np.mean(early_warning_hours), color='red', linestyle='-.', linewidth=2.5, label=f'Mean: {np.mean(early_warning_hours):.1f}h early')
plt.xlim(0, 150)
plt.xlabel('Advance Warning Hours Before Clinical Diagnosis', fontsize=12)
plt.ylabel('Patient Count', fontsize=12)
plt.title('Distribution of Early Warning Lead Time on Unseen Sepsis Cohort', fontsize=13, fontweight='bold')
plt.legend(frameon=True, fontsize=11)
plt.tight_layout()
plt.savefig(f"{figures_dir}/early_warning_distribution.png", dpi=300)
plt.show()

## 13. Game-Theoretic Model Interpretability via SHAP

In [ ]:
import shap

# Use TreeExplainer on a sample of the test set
explainer = shap.TreeExplainer(m_xgb)
sample_indices = np.random.RandomState(42).choice(len(X_test), size=1000, replace=False)
X_test_sample_df = pd.DataFrame(X_test[sample_indices], columns=feature_names)

shap_values = explainer.shap_values(X_test_sample_df)

plt.figure()
shap.summary_plot(shap_values, X_test_sample_df, show=False)
plt.title("SHAP Feature Impact: Sepsis Risk Drivers", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{figures_dir}/shap_summary.png", dpi=300, bbox_inches='tight')
plt.show()

## 14. Save Models & Final Checkpoint

In [ ]:
import joblib

joblib.dump(meta_learner, f"{project_dir}/stacking_meta_learner.pkl")
joblib.dump(m_xgb, f"{project_dir}/xgb_base_model.pkl")
joblib.dump(m_lgb, f"{project_dir}/lgb_base_model.pkl")
joblib.dump(m_rf, f"{project_dir}/rf_base_model.pkl")
joblib.dump(m_lr, f"{project_dir}/lr_base_model.pkl")
torch.save(m_dnn.state_dict(), f"{project_dir}/deepnet_weights.pt")
joblib.dump(scaler, f"{project_dir}/feature_scaler.pkl")

print("All models, scalers, and publication artifacts successfully saved to Google Drive!")